# Lab 01｜从一次 500 告警看 Agent Loop

只给目标和材料范围，让 Agent 自己寻找线索。重点看每次工具返回之后，它接下来做了什么。

## 核心代码：启动调查，并读取工具动作与返回信息

> 本讲重点看下面这 8 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 2 节，第 7 个单元格，第 11 至 12 行：**

```python
thread = lab.thread(sandbox=Sandbox.workspace_write)
result = investigate(thread, task)
```

`lab.thread()` 建立会话，`investigate()` 通过本讲辅助函数启动任务；模型在任务中选择工具并继续调查。

**公共准备，第 3 个单元格的 `events()`，第 59 至 64 行：**

```python
for wrapped in result.items:
    item = wrapped.root
    if item.type == "commandExecution":
        rows.extend([("采取的动作", item.command),
                     ("执行状态与退出码", f"{item.status.value} / {item.exit_code}"),
                     ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
```

这 6 行读取已完成任务的事件，把工具命令与返回信息配对。第 3 节的 `events(result)` 会展示它们；这里不是实时订阅。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `DEEPSEEK_API_KEY`；未设置时弹出隐藏输入框。模型通过原生 Responses 使用 DeepSeek Flash，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、请求观察和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(package + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex",))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("DEEPSEEK_API_KEY", "").strip():
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "01-agent-loop")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("执行状态与退出码", f"{item.status.value} / {item.exit_code}"),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))


def proxy_failure_hint(log_path):
    if not log_path.is_file():
        return None
    text = log_path.read_text(encoding="utf-8", errors="replace")
    if "AuthenticationError" in text or "401 Unauthorized" in text:
        return "模型服务拒绝了身份验证。请检查 DEEPSEEK_API_KEY 及其 API 权限；后续限流可能只是适配器进入冷却期，不能据此判断调查已执行。"
    if "RateLimitError" in text or "429 Too Many Requests" in text:
        return "模型服务或适配器限制了请求。请检查账号额度与服务状态；不要把重试结束当作调查完成。"
    return None


def investigate(thread, task):
    try:
        return run(thread, task, sandbox=Sandbox.workspace_write)
    except Exception as exc:
        hint = proxy_failure_hint(lab.runtime / "adapter.log")
        lab.close()
        if hint:
            raise RuntimeError(hint) from exc
        raise

print("公共准备完成。")


## 1. 查看本次材料｜必读必跑

告警、脱敏日志和代码放在同一个工作目录。这里展示给你看的内容，不会自动成为模型的输入；Agent 仍须自己读取文件。


In [ ]:
ROOT = lab.work
(ROOT / "inputs").mkdir()
(ROOT / "outputs").mkdir()
records = {
    "alert.md": """# HTTP 500 告警

- 时间：02:07
- 服务：payment-api
- 接口：POST /checkout
- 状态码：HTTP 500
- 错误率：8.4%
- 样例请求：req-7f3a
""",
    "logs/payment-api.log": """2026-08-24T02:06:58Z INFO request_id=req-7f39 path=/checkout status=200
2026-08-24T02:07:03Z INFO request_id=req-7f3a path=/checkout payload={"amount":29900}
2026-08-24T02:07:03Z ERROR request_id=req-7f3a status=500 error="KeyError: 'currency'"
Traceback (most recent call last):
  File "src/checkout.py", line 4, in create_order
    "currency": payload["currency"],
KeyError: 'currency'
""",
    "src/checkout.py": """def create_order(payload, repository):
    order = {
        "amount": payload["amount"],
        "currency": payload["currency"],
    }
    return repository.save(order)
""",
}

for name, text in records.items():
    path = ROOT / "inputs" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


input_snapshot = snapshot(ROOT / "inputs")
for name, content in input_snapshot.items():
    display(Markdown(f"### `inputs/{name}`\n```text\n{content.decode()}\n```"))


## 2. 启动调查｜必读必跑

`cwd` 指定工作目录；`workspace_write` 允许写报告，`deny_all` 拒绝扩权申请。本讲用文字要求保护输入，运行后检查是否改变；第 09 讲再落实目录级限制。

下面开始调用模型。不指定读取顺序，也不预告错误原因。


In [ ]:
task = """
请根据 inputs/ 中的告警、日志和代码，定位结账请求失败的最可能原因，
把调查结论写入 outputs/incident-report.md。

每项结论都要注明日志或代码位置。
没有证据的内容不要猜，也不要修改任何输入文件。
只使用这次提供的材料，不查询工作目录之外的文件或网络。
""".strip()


thread = lab.thread(sandbox=Sandbox.workspace_write)
result = investigate(thread, task)
print("Thread ID：", thread.id)
print("Turn ID：", result.id)


## 3. 沿着事件读一遍调查｜必读必跑

在相邻事件里找两件事：**刚获得的信息**是什么，**随后采取的动作**是什么。模型也可能合并读取文件；以实际发生的顺序为准。


In [ ]:
events(result)


## 4. 看报告，再看输入有没有变化｜必读必跑

报告引用能否回到上面的材料？默认只显示输入是否改变；有变化时再展开具体内容。


In [ ]:
report_path = ROOT / "outputs" / "incident-report.md"
display(Markdown(report_path.read_text() if report_path.exists() else "报告尚未生成，请回看实际事件。"))
compare_inputs(ROOT / "inputs", input_snapshot)


## 继续观察

对照工具事件中的命令和返回内容，找出一处“先得到线索，再改变下一步行动”的过程。如果这次模型合并读取了几个文件，如实记录，不要把它改写成逐步搜索。


## 从本次观察带走什么

找一条真实命令，把调用、退出码、返回内容和下一步放在一起。循环的价值在于新增观察能改变后续动作。再回到 req-7f3a：它能解释样例错误；若要解释全部 8.4%，还缺哪些请求？把未覆盖范围保留在结论中。

### 把观察讲成设计判断

我的关注点是支付告警能否形成可核查的调查链。我用请求编号把日志中的缺字段异常与代码访问位置连起来，让模型选择读取动作，工具提供实际观察。相比直接让模型猜根因，这样可以回查每个结论的来源。样例能解释这一条请求为何失败；整批 8.4% 错误的共同原因仍需更多请求和同时间窗数据。

**追问：读取了三个文件，为什么还不能证明整个告警的根因？** 三份材料围绕同一个请求，扩大的是该请求的证据链，没有扩大失败样本覆盖。应按请求分组检查相同堆栈、缺字段比例与发布时间，再决定能否推广这个解释。

讲述前回查本次请求与回执；结果以本次运行为准，示例措辞不增加实跑覆盖。


## 清理｜观察结束后必跑

关闭本讲的连接和观察服务。再次运行请从公共准备开始。


In [ ]:
lab.close()
